# <div align="center">Proyek Deep Learning</div>
<p align="center">Yohanes Kenny</p>
<p align="center">Benedict Hevel</p>
<p align="center">F Adiwidya Wirawan J</p>
<p align="center">Rayhandi Zulmi</p>
<p align="center">Farell Eugene</p>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
from pathlib import Path

base_path = 'dataset3/'

# 5 gambar pertama

5 gambar pertama dicek untuk memastikan warna, ukuran dan kualitas gambar.

In [2]:
# Lokasi file gambar

angry = base_path + 'angry_face'
happy = base_path + 'happy_face'
sad = base_path + 'sad_face'

angry_files = os.listdir(angry)
happy_files = os.listdir(happy)
sad_files = os.listdir(sad)

In [3]:
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

# angry
for i, img_file in enumerate(angry_files[:5]):
    img = Image.open(os.path.join(angry, img_file))
    axes[i].imshow(img)
    axes[i].set_title(img_file.split('.')[0])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [4]:
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

# happy
for i, img_file in enumerate(happy_files[:5]):
    img = Image.open(os.path.join(happy, img_file))
    axes[i].imshow(img)
    axes[i].set_title(img_file.split('.')[0])
    axes[i].axis('off')
    
plt.tight_layout()
plt.show()

In [5]:
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

# sad
for i, img_file in enumerate(sad_files[:5]):
    img = Image.open(os.path.join(sad, img_file))
    axes[i].imshow(img)
    axes[i].set_title(img_file.split('.')[0])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# PREPROCESSING

# Ubah ke dataframe

Gambar diubah ke dataframe terlebih dahulu agar dapat diproses dengan lebih mudah.

In [6]:
# Ada tiga jenis gambar, maka kategorinya dibagi tiga
categories = [
    'happy_face', 
    'sad_face', 
    'angry_face', 
]

# Ambil nama file gambar
all_paths = []

for category in categories:
    category_path = Path(base_path) / category
    
    paths = [path.parts[-2:] for path in category_path.rglob('*.*')]
    all_paths.extend(paths)

# Buat dataframe
df = pd.DataFrame(all_paths, columns=['Class', 'Images'])

# Urut berdasarkan class name
df = df.sort_values('Class', ascending=True)
df.reset_index(drop=True, inplace=True)

df

,Class,Images
0,angry_face,PrivateTest_8186613.jpg
1,angry_face,PrivateTest_55564530.jpg
2,angry_face,PrivateTest_54624150.jpg
3,angry_face,PrivateTest_54388384.jpg
4,angry_face,PrivateTest_54204181.jpg
...,...,...
1365,uncropped,8ea06887734fddc92e88484497516050.jpg
1366,uncropped,b3d0501772e7d3679a1abd5840e8b24d.jpg
1367,uncropped,b9488da2c9c5a7110f1c5f39a7832d80.jpg
1368,uncropped,f4d5fd4ed3bfd23da995088c630ab414.jpg


In [7]:
# Ambil info tentang jumlah gambar yang ada
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1370 entries, 0 to 1369
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Class   1370 non-null   object
 1   Images  1370 non-null   object
dtypes: object(2)
memory usage: 21.5+ KB


In [8]:
# Buat dataframe dari setiap class gambar

happy_df = df[df['Class'] == 'happy_face']
angry_df = df[df['Class'] == 'angry_face']
sad_df = df[df['Class'] == 'sad_face']
neutral_df = df[df['Class'] == 'neutral_face']

In [9]:
print(happy_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 418 entries, 415 to 832
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Class   418 non-null    object
 1   Images  418 non-null    object
dtypes: object(2)
memory usage: 9.8+ KB
None


In [10]:
print(sad_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 419 entries, 833 to 1251
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Class   419 non-null    object
 1   Images  419 non-null    object
dtypes: object(2)
memory usage: 9.8+ KB
None


In [11]:
print(angry_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 415 entries, 0 to 414
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Class   415 non-null    object
 1   Images  415 non-null    object
dtypes: object(2)
memory usage: 9.7+ KB
None


# Hapus gambar duplikat atau korup

In [12]:
import hashlib

valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
valid_images = []

for i, row in df.iterrows():
    img_path = os.path.join(base_path, row['Class'], row['Images'])
    ext = os.path.splitext(img_path)[1].lower()
    
    # Untuk cek apakah format file valid?
    if ext not in valid_extensions:
        print(f"Skipping unsupported format: {img_path}")
        continue
    
    try:
        with Image.open(img_path) as img:
            img.verify()
        valid_images.append(row.to_dict())
    except Exception as e:
        print(f"Skipping corrupt or unreadable image: {img_path} ({e})")

# Buat ulang DataFrame hanya dengan gambar valid
df = pd.DataFrame(valid_images).reset_index(drop=True)
print(f"Total gambar valid (format OK & tidak corrupt): {len(df)}")

# Untk deteksi duplikat menggunakan hash MD5
def calculate_md5(img_path):
    with open(img_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# Hitung hash pada setiap gambar
df['MD5_Hash'] = df.apply(
    lambda row: calculate_md5(os.path.join(base_path, row['Class'], row['Images'])),
    axis=1
)

# Hapus gambar duplikat berdasarkan hash
before_dedup = len(df)
df = df.drop_duplicates(subset='MD5_Hash').reset_index(drop=True)
after_dedup = len(df)

print(f"Total gambar unik setelah menghapus duplikat: {after_dedup} (dihapus {before_dedup - after_dedup})")

df.head()

Skipping corrupt or unreadable image: dataset3/uncropped\246365c4e7bd4439382590ef4a149e63.jpg ([Errno 2] No such file or directory: 'C:\\Users\\Francis\\Documents\\-- MY SHIT\\-- Coolidge\\Sem 5\\Deep Learning\\Proyek\\dataset3\\uncropped\\246365c4e7bd4439382590ef4a149e63.jpg')
Skipping corrupt or unreadable image: dataset3/uncropped\5f575684adc3a0ff53f26641f33e65ae.jpg ([Errno 2] No such file or directory: 'C:\\Users\\Francis\\Documents\\-- MY SHIT\\-- Coolidge\\Sem 5\\Deep Learning\\Proyek\\dataset3\\uncropped\\5f575684adc3a0ff53f26641f33e65ae.jpg')
Skipping corrupt or unreadable image: dataset3/uncropped\3c7210c407047bf26b825c7665b8c529.jpg ([Errno 2] No such file or directory: 'C:\\Users\\Francis\\Documents\\-- MY SHIT\\-- Coolidge\\Sem 5\\Deep Learning\\Proyek\\dataset3\\uncropped\\3c7210c407047bf26b825c7665b8c529.jpg')
Skipping corrupt or unreadable image: dataset3/uncropped\3e5d95f7767a9577fe4299c1c4cef29a.jpg ([Errno 2] No such file or directory: 'C:\\Users\\Francis\\Documents\

,Class,Images,MD5_Hash
0,angry_face,PrivateTest_8186613.jpg,8f5b9a39c0caf723e0ea40e02dbdc881
1,angry_face,PrivateTest_55564530.jpg,bd46cea0345eaac4aecb14f51185099d
2,angry_face,PrivateTest_54624150.jpg,503bc8ce972fa42cfa037df04369f64c
3,angry_face,PrivateTest_54388384.jpg,d45beb8f32f33c9386c0753ce166ce66
4,angry_face,PrivateTest_54204181.jpg,b1613ada612e4ea9a01dae5a516dfcb8


# EXPLORATORY DATA ANALYSIS

# Dimensi gambar

Dimensi gambar diambil untuk mengecek aspect ratio dan mendeteksi outlier.

In [13]:
categories = [
    'happy_face', 
    'sad_face', 
    'angry_face', 
]

colors = {
    'happy_face': 'skyblue', 
    'sad_face': 'salmon', 
    'angry_face': 'lightgreen',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, category in enumerate(categories):
    category_path = os.path.join(base_path, category)
    image_files = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    image_dimensions = [Image.open(os.path.join(category_path, img_file)).size for img_file in image_files]
    widths, heights = zip(*image_dimensions)

    axes[0, i].hist(widths, bins=30, color=colors[category], edgecolor='black')
    axes[0, i].set_title(f'{category} - Widths')
    axes[0, i].set_xlabel('Width')
    axes[0, i].set_ylabel('Number of Images')

    axes[1, i].hist(heights, bins=30, color=colors[category], edgecolor='black')
    axes[1, i].set_title(f'{category} - Heights')
    axes[1, i].set_xlabel('Height')
    axes[1, i].set_ylabel('Number of Images')

plt.tight_layout()
plt.show()

# Class distribution

Class distribution dicek untuk melihat jika ada class imbalance atau tidak.

In [14]:
ax = sns.countplot(data=df, 
                   x = 'Class',
                   palette='Set1', 
                   order=df['Class'].value_counts().index)

for container in ax.containers:
    ax.bar_label(container, fmt="%d")
    
plt.show()

C:\Users\Francis\AppData\Local\Temp\ipykernel_13112\507861827.py:1: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(data=df,


# Distribusi warna RGB

Digunakan untuk mendeteksi domain bias dan overfitting

In [15]:
colors = {
    "happy_face": "gold", 
    "angry_face": "red", 
    "sad_face": "blue",
}

def sample_pixels_from_folder(folder, max_per_image=10000):
    samples = []
    # Melakukan iterasi pada setiap file di dalam folder
    for filename in os.listdir(folder):
        # Memastikan file adalah format gambar yang didukung
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            img_path = os.path.join(folder, filename)
            
            # Membuka gambar dan mengubahnya ke format RGB
            img = Image.open(img_path).convert("RGB")
            arr = np.array(img)
            
            # Mengubah bentuk array menjadi daftar panjang berisi nilai RGB (N, 3)
            pixels = arr.reshape(-1, 3)
            
            # Jika jumlah piksel melebihi batas maksimal, ambil sampel secara acak
            if len(pixels) > max_per_image:
                idx = np.random.choice(len(pixels), size=max_per_image, replace=False)
                pixels = pixels[idx]
            
            samples.append(pixels)
            
    # Menggabungkan semua sampel piksel menjadi satu array tunggal
    if samples:
        return np.vstack(samples)
    return np.array([])

# Dict untuk menyimpan data piksel berdasarkan kategori
pixels_by_category = {}

# Melakukan ekstraksi piksel untuk setiap kategori (happy, sad, angry)
for category in categories:
    folder_path = os.path.join(base_path, category)
    pixels = sample_pixels_from_folder(folder_path)
    pixels_by_category[category] = pixels
    print(f"{category}: berhasil mengambil {len(pixels)} sampel piksel")

# Visualisasi
plt.figure(figsize=(12, 6))

for category in categories:
    pixels = pixels_by_category[category]
    if pixels.size == 0:
        continue
    
    # Menghitung rata-rata intensitas (brightness) dari nilai RGB untuk setiap piksel
    intensities = pixels.mean(axis=1)
    
    # Membuat kurva distribusi densitas (KDE Plot) untuk setiap kategori
    sns.kdeplot(intensities, color=colors[category], label=category, fill=True, alpha=0.3)

# Memberikan label dan judul pada grafik
plt.title("Distribusi RGB")
plt.xlabel("Rata-rata intensitas pixel (0–255)")
plt.ylabel("Densitas")
plt.legend(title="Emosi")
plt.show()

happy_face: berhasil mengambil 1270912 sampel piksel
sad_face: berhasil mengambil 1280912 sampel piksel
angry_face: berhasil mengambil 1237312 sampel piksel


# Aspect ratio

Apakah semua gambar memiliki aspect ratio 1:1? Jika tidak, maka gambar harus dipotong nanti

In [16]:
data = []

# Melakukan iterasi untuk setiap kategori label (nama folder)
for label in [
    "happy_face", 
    "sad_face", 
    "angry_face", 
    ]:
    # Menggabungkan path dasar dengan nama folder label
    folder = os.path.join(base_path, label)
    
    # Melakukan iterasi untuk setiap file di dalam folder tersebut
    for filename in os.listdir(folder):
        # Memastikan file adalah gambar berdasarkan ekstensinya
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")):
            img_path = os.path.join(folder, filename)
            try:
                # Membuka gambar menggunakan library PIL (Image)
                with Image.open(img_path) as img:
                    width, height = img.size
                    # Menghitung aspect ratio (lebar dibagi tinggi)
                    aspect_ratio = width / height
                    # Menyimpan label dan hasil perhitungan ke dalam list data
                    data.append({"label": label, "aspect_ratio": aspect_ratio})
            except Exception as e:
                # Menampilkan pesan error jika gambar rusak atau tidak bisa dibuka
                print(f"Melewati {img_path}: {e}")

# Mengonversi list data menjadi DataFrame pandas agar mudah diolah
df_aspect_ratio = pd.DataFrame(data)



# Visualisasi
plt.figure(figsize=(8, 5))

# Membuat plot estimasi kepadatan kernel (KDE) untuk melihat distribusi aspect ratio
sns.kdeplot(
    data=df_aspect_ratio, 
    x="aspect_ratio", 
    hue="label",          # Membedakan warna berdasarkan label emosi
    fill=True,            # Memberi warna isi di bawah kurva
    common_norm=False,    # Menormalkan setiap kurva secara independen
    alpha=0.6             # Mengatur transparansi warna
)

# Menambahkan judul dan label keterangan sumbu
plt.title("Densitas aspect ratio berdasarkan kategori emosi")
plt.xlabel("Aspect ratio (lebar / panjang)")
plt.ylabel("Densitas")

# Menampilkan plot hasil akhir
plt.show()

C:\Users\Francis\AppData\Local\Temp\ipykernel_13112\521031735.py:38: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(


# Kecerahan gambar

Kita ingin melihat distribusi kecerahan gambar untuk mengidentifikasi isu exposure

In [17]:
folders = {
    "happy": base_path + "happy_face",
    "sad": base_path + "sad_face",
    "angry": base_path + "angry_face",
}

# Fungsi untuk menghitung rata-rata tingkat kecerahan (brightness) dari sebuah gambar
def image_brightness(img_path):
    # Membuka gambar dan mengubahnya ke mode grayscale ("L")
    img = Image.open(img_path).convert("L")
    # Mengubah data gambar menjadi array numpy
    arr = np.array(img)
    # Mengembalikan nilai rata-rata dari seluruh piksel (0-255)
    return arr.mean()

# Inisialisasi dictionary untuk menyimpan data kecerahan setiap label
brightness_data = {}

# Melakukan iterasi pada setiap folder kategori
for label, folder in folders.items():
    brightness_data[label] = []
    # Membaca semua file di dalam folder
    for filename in os.listdir(folder):
        # Memastikan file adalah format gambar yang didukung
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            try:
                # Menggabungkan path folder dengan nama file
                path = os.path.join(folder, filename)
                # Menambahkan hasil hitung kecerahan ke dalam list sesuai labelnya
                brightness_data[label].append(image_brightness(path))
            except:
                # Lewati file jika terjadi error saat pemrosesan
                continue

# Menyusun ulang data ke dalam bentuk list of dictionaries agar mudah diubah ke DataFrame
data = []
for label, values in brightness_data.items():
    for val in values:
        data.append({"label": label, "brightness": val})

# Mengimpor pandas dan membuat DataFrame dari data yang terkumpul
import pandas as pd
df_brightness = pd.DataFrame(data)

# Visualisasi
plt.figure(figsize=(10,6))
# Membuat plot KDE (Kernel Density Estimate) untuk melihat distribusi kecerahan
sns.kdeplot(data=df_brightness, x="brightness", hue="label", fill=True, common_norm=False, alpha=0.5)

# Memberikan judul dan label pada grafik
plt.title("Distribusi kecerahan berdasarkan emosi")
plt.xlabel("Kecerahan (0-255)")
plt.ylabel("Densitas")
plt.show()

# Ukuran file

Digunakan untuk melihat gambar yang masih korup dan untuk mendeteksi file yang terlalu kecil (yang berarti kualitas file tersebut buruk).

In [18]:
data = []

# Melakukan iterasi untuk setiap label/kategori dalam daftar 'categories'
for label in categories:
    # Menggabungkan path dasar dengan nama label untuk mendapatkan path folder kategori
    folder_path = os.path.join(base_path, label)
    
    # Melakukan iterasi untuk setiap file yang ada di dalam folder tersebut
    for filename in os.listdir(folder_path):
        # Membuat path lengkap ke file tertentu
        file_path = os.path.join(folder_path, filename)
        
        # Memastikan bahwa objek tersebut adalah file (bukan folder)
        if os.path.isfile(file_path):
            # Mendapatkan ukuran file dan mengubahnya dari Byte ke KB
            size_kb = os.path.getsize(file_path) / 1024
            
            # Memasukkan informasi kelas, nama file, dan ukurannya ke dalam list data
            data.append({"class": label, "filename": filename, "size_kb": size_kb})

# Mengonversi list 'data' menjadi DataFrame pandas agar mudah diolah
df_file_size = pd.DataFrame(data)

# Menampilkan 5 baris pertama dari DataFrame untuk pengecekan cepat
df_file_size.head()

,class,filename,size_kb
0,happy_face,000c6931aab182e6800e9a63a6beb673.jpg,7.191406
1,happy_face,041558267adb4816babe5aef2d26a46f.jpg,5.426758
2,happy_face,0b8af5e8bf7a230dd4e13ec67a4bf283.jpg,8.900391
3,happy_face,19230244c1a62251c3f10fe6dd4eeb7a.jpg,2.595703
4,happy_face,1b92345b60c29ee6f1e80f9904c7029e.jpg,9.451172


In [19]:
df_file_size.groupby("class")["size_kb"].describe()

,count,mean,std,min,25%,50%,75%,max
class,,,,,,,,
angry_face,415.0,9.432789,71.779245,1.013672,1.459961,1.583984,1.690430,858.608398
happy_face,418.0,1.962631,1.391735,0.913086,1.448975,1.556641,1.659912,9.451172
sad_face,419.0,5.869637,41.141739,0.965820,1.436035,1.534180,1.655273,588.923828


In [20]:
# Visualisasi

plt.figure(figsize=(8, 5))
df_file_size.boxplot(column="size_kb", by="class")
plt.title("Distribusi ukuran file berdasarkan emosi")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Ukuran file (kb)")
plt.show()

<Figure size 800x500 with 0 Axes>

# Konversi ke grayscale dan CLAHE

Gambar yang belum grayscale dikonversi ke grayscale agar:
1. tidak memakan banyak memori saat training dan modeling; dan
2. training lebih cepat; dan
3. mencegah overfitting.

Pada dasarnya, CLAHE memperbaiki kualitas gambar agar benda di dalam gambar lebih mudah dilihat. CLAHE mengurangi noise dalam gambar dan merapikan pencahayaan dalam gambar.

In [21]:
import cv2

def clahe(pil_img):
    img = np.array(pil_img)

    clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe_obj.apply(img)

    return Image.fromarray(img)

In [22]:
converted_gray_count = 0

for i, row in df.iterrows():
    img_path = os.path.join(base_path, row['Class'], row['Images'])
    try:
        with Image.open(img_path) as img:
            if img.mode != 'L':  # Jika bukan grayscale
                img = img.convert('L')  # Ubah ke grayscale
                img = clahe(img)
                img.save(img_path)
                converted_gray_count += 1
    except Exception as e:
        print(f"konversi gambar: {img_path} ({e})")

print(f"Total gambar yang dikonversi ke grayscale: {converted_gray_count}")

Total gambar yang dikonversi ke grayscale: 0


# Resizing

Resizing dilakukan untuk mendapatkan ukuran input yang fix dan meningkatkan efisiensi.

In [23]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Dimensi standar untuk semua gambar
IMG_WIDTH, IMG_HEIGHT = 48, 48

X = []
y = []

for i, row in df.iterrows():
    # Menggabungkan path folder utama, nama kelas, dan nama file untuk mendapatkan path lengkap gambar
    img_path = os.path.join(base_path, row['Class'], row['Images'])
    
    try:
        # Memuat gambar dari path, mengubah ukurannya menjadi 48x48, dan mengubahnya ke skala abu-abu (grayscale)
        img = load_img(img_path, target_size=(IMG_WIDTH, IMG_HEIGHT), color_mode='grayscale')
        
        # Mengonversi objek gambar menjadi array numpy agar bisa diproses secara matematis
        img_array = img_to_array(img)
        
        # Normalisasi: Mengubah nilai piksel (0-255) menjadi rentang 0-1 untuk mempercepat konvergensi model
        img_array = img_array / 255.0
        
        # Memasukkan array gambar ke list X dan label kelasnya ke list y
        X.append(img_array)
        y.append(row['Class'])
        
    except Exception as e:
        # Error jika ada gambar yang rusak atau path yang salah
        print(f"Error loading image {img_path}: {e}")

In [24]:
X = np.array(X)
y = np.array(y)

print(f"Shape data gambar (X): {X.shape}")
print(f"Shape label (y): {y.shape}")

Shape data gambar (X): (1249, 48, 48, 1)
Shape label (y): (1249,)


# Encoding

In [25]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

print("Mapping kelas:", dict(zip(le.classes_, range(len(le.classes_)))))

Mapping kelas: {'angry_face': 0, 'happy_face': 1, 'sad_face': 2}


# Train-test split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"Data Latih: {X_train.shape}")
print(f"Data Uji: {X_test.shape}")

Data Latih: (999, 48, 48, 1)
Data Uji: (250, 48, 48, 1)


# Modeling

In [27]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

def create_cnn_model(input_shape=(48, 48, 1), num_classes=3):
    # Menggunakan Sequential karena data mengalir searah dari input ke output secara linear.
    model = Sequential()

    # Ekstraksi Fitur Dasar
    # Conv2D: 32 filter digunakan untuk menangkap fitur sederhana seperti tepi dan garis.
    # 'padding=same' menjaga ukuran output tetap sama dengan input agar informasi di pinggir gambar tidak hilang.
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape, padding='same'))
    # BatchNormalization: Menormalkan aktivasi di setiap batch. Ini memungkinkan penggunaan learning rate 
    # yang lebih tinggi dan mengurangi ketergantungan pada inisialisasi bobot yang sangat presisi.
    model.add(BatchNormalization())
    # MaxPooling: Mengambil nilai maksimal dalam jendela 2x2. Ini mereduksi resolusi spasial sebesar 50%,
    # membuat model lebih 'invariant' terhadap pergeseran kecil pada objek di dalam gambar.
    model.add(MaxPooling2D(pool_size=(2, 2)))
    # Dropout 0.25: Mematikan 25% neuron secara acak untuk memaksa jaringan mencari jalur alternatif 
    # dalam mengenali pola, sehingga tidak hanya bergantung pada satu set neuron (mencegah overfitting).
    model.add(Dropout(0.25))

    # Fitur Menengah
    # Jumlah filter dinaikkan menjadi 64 karena ruang pencarian fitur menjadi lebih kompleks 
    # setelah dimensi spasial diperkecil oleh pooling sebelumnya.
    model.add(Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Fitur Kompleks
    # 128 filter memungkinkan model mengenali pola yang lebih abstrak (seperti bentuk wajah atau tekstur spesifik).
    model.add(Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Bagian Klasifikasi (FCL)
    # Flatten: Mengubah tensor 3D (height, width, channels) menjadi vektor 1D. 
    # Ini diperlukan karena Dense layer hanya menerima input dalam bentuk vektor tunggal.
    model.add(Flatten())
    
    # Dense (128): Layer tersembunyi untuk mempelajari kombinasi fitur non-linear dari hasil konvolusi.
    # L2 Regularization: Memberikan penalti pada bobot yang terlalu besar (weight decay). 
    # Ini menjaga agar model tetap sederhana dan tidak "menghafal" data training secara berlebihan.
    model.add(Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(BatchNormalization())
    # Dropout lebih tinggi (0.5) digunakan di layer Dense karena area ini paling rentan terhadap overfitting
    # mengingat jumlah parameter yang sangat besar di bagian fully connected.
    model.add(Dropout(0.5))

    # Output Layer:
    # Softmax digunakan untuk mengubah output logit menjadi nilai probabilitas (0 sampai 1) 
    # yang jika dijumlahkan totalnya adalah 1, sangat cocok untuk klasifikasi multi-kelas (3 kelas).
    model.add(Dense(num_classes, activation='softmax'))

    return model

INPUT_SHAPE = (48, 48, 1)
NUM_CLASSES = 3

model = create_cnn_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)

# Strategi Kompilasi:
# Adam: Optimizer adaptif yang menggabungkan keunggulan RMSProp dan Momentum. 
# Sangat efektif untuk masalah pengenalan gambar karena cepat mencapai konvergensi.
# Categorical Crossentropy: Fungsi loss standar untuk klasifikasi di mana satu input hanya memiliki satu label kelas.
model.compile(
    optimizer=Adam(learning_rate=0.001), 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()

c:\Users\Francis\anaconda3\envs\Data\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 684,419 (2.61 MB)

 Trainable params: 683,715 (2.61 MB)

 Non-trainable params: 704 (2.75 KB)

# Model training

In [28]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32
)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 9s 89ms/step - accuracy: 0.3541 - loss: 2.0993 - val_accuracy: 0.3360 - val_loss: 1.6422
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.4435 - loss: 1.6178 - val_accuracy: 0.3320 - val_loss: 2.1026
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.4832 - loss: 1.4850 - val_accuracy: 0.3320 - val_loss: 2.4684
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.5203 - loss: 1.3897 - val_accuracy: 0.3320 - val_loss: 1.9783
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.5537 - loss: 1.3044 - val_accuracy: 0.3400 - val_loss: 1.9606
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.5828 - loss: 1.2087 - val_accuracy: 0.3360 - val_loss: 4.0187
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.5552 - loss: 1.2404 - val_accuracy: 0.3520 - val_loss: 3.9950
Epoch 8/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.6098 - loss: 1.1578 - val_accuracy: 0.3360 - v

# Evaluation

In [29]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Data histori pelatihan
acc = history.history['accuracy']          # Akurasi pelatihan
val_acc = history.history['val_accuracy']  # Akurasi validasi
loss = history.history['loss']             # Loss/kerugian pelatihan
val_loss = history.history['val_loss']     # Loss/kerugian validasi
epochs_range = range(len(acc))             # Membuat rentang angka berdasarkan jumlah epoch

# Visualisasi akurasi dan loss
plt.figure(figsize=(12, 4))

# Grafik akurasi
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Grafik loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# Evaluasi model pada data test
print("Mengevaluasi pada data test...")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# Prediksi label
y_pred_probs = model.predict(X_test)             # Prediksi probabilitas dari model
y_pred_classes = np.argmax(y_pred_probs, axis=1) # Mengambil indeks kelas dengan probabilitas tertinggi
y_true = np.argmax(y_test, axis=1)               # Mengambil indeks kelas asli dari data uji

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Angry', 'Happy', 'Sad', 'Neutral'], 
            yticklabels=['Angry', 'Happy', 'Sad', 'Neutral'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=['Angry', 'Happy', 'Sad']))

Mengevaluasi pada data test...
Test Accuracy: 59.60%
Test Loss: 1.5189
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step



Classification Report:
              precision    recall  f1-score   support

       Angry       0.62      0.43      0.51        83
       Happy       0.66      0.70      0.68        84
         Sad       0.52      0.65      0.58        83

    accuracy                           0.60       250
   macro avg       0.60      0.60      0.59       250
weighted avg       0.60      0.60      0.59       250



# Save the model

In [30]:
model.save('emotion_model_v8.h5')